## 第二部分：数据清洗与变量构造


## 2.1 基础清洗要求

| 清洗项目 | 具体要求 |
| --- |  --- |
| 编码与读取 | 尝试识别文件编码，例如 、 或 ；若读取失败，说明原因和处理方式`utf-8-sig``gbk``gb18030` |
| 主键统一 | 将股票代码统一为 6 位字符串，记为 ；将会计年度统一为整数型，记为`code``year` |
| 日期处理 | 将报表日期、上市日期等日期变量统一为 格式`datetime64` |
| 年度样本 | 保留 2000 年至今的年度观测；如果原始数据起始年份较晚，以实际年份为准 |
| 重复值处理 | 检查 是否重复；若重复，说明保留规则`code-year` |
| 缺失值检测 | 统计核心变量的缺失数量和缺失比例，并解释可能原因 |
| 数值型转换 | 将金额、比例、股权比例等变量转换为数值型 |
| 异常值检查 | 检查分母为 0、负资产、负权益、比例异常等问题，并说明处理方式 |

In [9]:
# ==============================================
# 功能：2.1 基础清洗要求
# 编码识别、主键统一、日期处理、重复值、缺失值、数值转换、异常值检查
# ==============================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("🧹 2.1 基础清洗要求")
print("=" * 60)

# ---------- 读取三个原始数据源 ----------
print("\n🔄 读取原始数据...")

# 1）财务数据
fin_raw = pd.read_excel("data/raw/CSMAR/data_raw_zip/跨表查询_沪深京股票(年频).xlsx")
fin_rename = {}
for col in fin_raw.columns:
    if col == "code": fin_rename[col] = "code"
    elif "enddate" in col.lower(): fin_rename[col] = "end_date"
    elif "A001100000" in col: fin_rename[col] = "total_asset"
    elif "A001101000" in col: fin_rename[col] = "current_asset"
    elif "A002000000" in col: fin_rename[col] = "total_liability"
    elif "A002101000" in col: fin_rename[col] = "current_liability"
    elif "A002200000" in col: fin_rename[col] = "noncurrent_liability"
    elif "A002206000" in col: fin_rename[col] = "long_term_borrow"
    elif "A002207000" in col: fin_rename[col] = "short_term_borrow"
    elif "A003101000" in col: fin_rename[col] = "revenue"
    elif "A003105000" in col: fin_rename[col] = "net_profit"
    elif "A003102000" in col: fin_rename[col] = "cash"
    elif "A003000000" in col: fin_rename[col] = "equity"
    elif "A004000000" in col: fin_rename[col] = "operating_cf"
fin_raw = fin_raw.rename(columns=fin_rename)
print(f"   财务数据：{fin_raw.shape[0]} 行 × {fin_raw.shape[1]} 列")

# 2）治理数据（列名是英文 Stkcd/accper）
gov_raw = pd.read_excel("data/raw/CSMAR/data_raw_zip/常用变量查询（年度）.xlsx", header=0)
gov_raw = gov_raw.iloc[1:].reset_index(drop=True)
gov_raw = gov_raw.rename(columns={
    "Stkcd": "code", "accper": "year",
    "Shrcr1": "Top1", "Shrhfd5": "HHI5", "Shrz": "Top2_to_Top1"
})
gov_raw = gov_raw[gov_raw["code"] != "股票代码"]
print(f"   治理数据：{gov_raw.shape[0]} 行 × {gov_raw.shape[1]} 列")

# 3）公司基本信息
info_raw = pd.read_excel("data/raw/CSMAR/data_raw_zip/STK_LISTEDCOINFOANL.xlsx")
info_rename = {}
for col in info_raw.columns:
    if "symbol" in col.lower(): info_rename[col] = "code"
    elif "listingdate" in col.lower(): info_rename[col] = "listing_date"
    elif "industrycode" in col.lower(): info_rename[col] = "industry_code"
    elif "industryname" in col.lower(): info_rename[col] = "industry_name"
    elif "shortname" in col.lower(): info_rename[col] = "stknme"
    elif "province" in col.lower(): info_rename[col] = "province"
info_raw = info_raw.rename(columns=info_rename)
print(f"   公司信息：{info_raw.shape[0]} 行 × {info_raw.shape[1]} 列")

# ---------- 2.1.1 主键统一 ----------
print("\n🗂️ 2.1.1 主键统一：code=6位字符串，year=整数...")

for df in [fin_raw, gov_raw, info_raw]:
    if "code" in df.columns:
        df["code"] = df["code"].astype(str).str.strip().str.zfill(6)

fin_raw["year"] = pd.to_datetime(fin_raw["end_date"], errors="coerce").dt.year
fin_raw["year"] = fin_raw["year"].astype("Int64")
gov_raw["year"] = pd.to_numeric(gov_raw["year"], errors="coerce").astype("Int64")
print("   ✅ 主键统一完成")

# ---------- 2.1.2 日期处理 ----------
print("\n🗂️ 2.1.2 日期处理：统一为 datetime64 格式...")

fin_raw["end_date"] = pd.to_datetime(fin_raw["end_date"], errors="coerce")
info_raw["listing_date"] = pd.to_datetime(info_raw["listing_date"], errors="coerce")
info_raw["list_year"] = info_raw["listing_date"].dt.year
print(f"   fin_raw['end_date'] dtype: {fin_raw['end_date'].dtype}")
print(f"   info_raw['listing_date'] dtype: {info_raw['listing_date'].dtype}")
print("   ✅ 日期格式统一完成（datetime64）")

# ---------- 2.1.3 缺失值检测 ----------
print("\n🗂️ 2.1.3 缺失值检测...")

def missing_table(df, name):
    print(f"\n   【{name}】核心变量缺失情况：")
    cols = [c for c in df.columns if c in [
        "code","year","total_asset","total_liability","current_liability",
        "net_profit","equity","cash","Top1","HHI5","listing_date"
    ]]
    print(f"   {'变量名':<22} {'非空数':>8} {'缺失数':>8} {'缺失率':>8}")
    print(f"   {'-'*50}")
    for col in cols:
        n = len(df)
        n_miss = df[col].isna().sum()
        pct = n_miss / n * 100
        flag = "⚠️" if pct > 5 else "  "
        print(f"   {flag}{col:<20} {n - n_miss:>8} {n_miss:>8} {pct:>7.2f}%")

missing_table(fin_raw, "财务数据")
missing_table(gov_raw, "治理数据")
missing_table(info_raw, "公司信息")
print("""
   📝 缺失原因说明：
   - 财务数据：CSMAR 本身从 2011 年才开始有财务报表，2000-2010年为天然缺失
   - 治理数据：部分公司未在 CSMAR 更新股权集中度数据
   - 上市日期：公司基本信息表仅保留最新状态，历史变动未记录""")

# ---------- 2.1.4 重复值处理 ----------
print("\n🗂️ 2.1.4 重复值处理：检查 code-year 重复...")

def dedup_report(df, name, subset=["code", "year"]):
    n_dup = df.duplicated(subset=subset).sum()
    n_before = df.shape[0]
    if n_dup > 0:
        print(f"   ⚠️  {name} 发现 {n_dup} 个重复，保留第一条...")
        df = df.drop_duplicates(subset=subset)
        print(f"   删重后：{n_before} → {df.shape[0]} 行")
    else:
        print(f"   ✅ {name} 无重复 code-year")
    return df

fin_raw = dedup_report(fin_raw, "财务数据")
gov_raw = dedup_report(gov_raw, "治理数据")

# ---------- 2.1.5 数值型转换 ----------
print("\n🗂️ 2.1.5 数值型转换：将金额、比例等转换为数值型...")

num_cols = [
    "total_asset","current_asset","total_liability","current_liability",
    "noncurrent_liability","long_term_borrow","short_term_borrow",
    "revenue","net_profit","cash","equity","operating_cf"
]
for col in num_cols:
    if col in fin_raw.columns:
        fin_raw[col] = pd.to_numeric(fin_raw[col], errors="coerce")

for col in ["Top1","HHI5","Top2_to_Top1"]:
    if col in gov_raw.columns:
        gov_raw[col] = pd.to_numeric(gov_raw[col], errors="coerce")

print("   ✅ 数值型转换完成")

# ---------- 2.1.6 异常值检查 ----------
print("\n🗂️ 2.1.6 异常值检查...")

n_zero = (fin_raw["total_asset"] == 0).sum()
if n_zero > 0:
    print(f"   ⚠️  发现 {n_zero} 条总资产为0，删除...")
    fin_raw = fin_raw[fin_raw["total_asset"] != 0]
else:
    print("   ✅ 总资产无非零异常")

n_neg_a = (fin_raw["total_asset"] < 0).sum()
n_neg_l = (fin_raw["total_liability"] < 0).sum()
n_neg_e = (fin_raw["equity"] < 0).sum()
if n_neg_a > 0: fin_raw = fin_raw[fin_raw["total_asset"] >= 0]; print(f"   ⚠️  删除 {n_neg_a} 条负资产")
if n_neg_l > 0: fin_raw = fin_raw[fin_raw["total_liability"] >= 0]; print(f"   ⚠️  删除 {n_neg_l} 条负负债")
if n_neg_e > 0: fin_raw = fin_raw[fin_raw["equity"] >= 0]; print(f"   ⚠️  删除 {n_neg_e} 条负权益")
if n_neg_a == 0 and n_neg_l == 0 and n_neg_e == 0:
    print("   ✅ 无负资产、负负债、负权益")

fin_raw["_lev"] = fin_raw["total_liability"] / fin_raw["total_asset"]
n_ab = ((fin_raw["_lev"] > 1) | (fin_raw["_lev"] < 0)).sum()
if n_ab > 0:
    print(f"   ⚠️  发现 {n_ab} 条杠杆率异常（>1或<0），删除...")
    fin_raw = fin_raw[(fin_raw["_lev"] <= 1) & (fin_raw["_lev"] >= 0)]
    fin_raw.drop(columns=["_lev"], inplace=True)
else:
    print("   ✅ 杠杆率无异常（均在[0,1]范围内）")
    fin_raw.drop(columns=["_lev"], inplace=True, errors="ignore")

print("   ✅ 异常值检查完成")

# ---------- 2.1.7 年度样本 ----------
print("\n🗂️ 2.1.7 年度样本：保留 2000 年至今...")

fin_y = sorted(fin_raw["year"].dropna().unique())
gov_y = sorted(gov_raw["year"].dropna().unique())
actual = min(min(fin_y), min(gov_y))
print(f"   财务起始：{min(fin_y)} 年，治理起始：{min(gov_y)} 年")
print(f"   以实际最早年份 {actual} 年为准")

b = fin_raw.shape[0]
fin_raw = fin_raw[fin_raw["year"] >= 2000]
print(f"   财务：{b} → {fin_raw.shape[0]} 行")

b = gov_raw.shape[0]
gov_raw = gov_raw[gov_raw["year"] >= 2000]
print(f"   治理：{b} → {gov_raw.shape[0]} 行")

print("   ✅ 年度样本筛选完成")

print("\n" + "=" * 60)
print("✅ 2.1 基础清洗完成！")
print("=" * 60)
print(f"   财务：{fin_raw.shape[0]} 行 × {fin_raw.shape[1]} 列")
print(f"   治理：{gov_raw.shape[0]} 行 × {gov_raw.shape[1]} 列")
print(f"   公司信息：{info_raw.shape[0]} 行 × {info_raw.shape[1]} 列")


🧹 2.1 基础清洗要求

🔄 读取原始数据...
   财务数据：81665 行 × 32 列
   治理数据：61457 行 × 33 列
   公司信息：64173 行 × 40 列

🗂️ 2.1.1 主键统一：code=6位字符串，year=整数...
   ✅ 主键统一完成

🗂️ 2.1.2 日期处理：统一为 datetime64 格式...
   fin_raw['end_date'] dtype: datetime64[ns]
   info_raw['listing_date'] dtype: datetime64[ns]
   ✅ 日期格式统一完成（datetime64）

🗂️ 2.1.3 缺失值检测...

   【财务数据】核心变量缺失情况：
   变量名                         非空数      缺失数      缺失率
   --------------------------------------------------
     code                    81665        0    0.00%
   ⚠️total_asset             53917    27748   33.98%
   ⚠️current_liability       45611    36054   44.15%
   ⚠️total_liability         54714    26951   33.00%
   ⚠️cash                    54575    27090   33.17%
   ⚠️net_profit              54714    26951   33.00%
   ⚠️equity                  54714    26951   33.00%
     year                    81662        3    0.00%

   【治理数据】核心变量缺失情况：
   变量名                         非空数      缺失数      缺失率
   --------------------------------------------------
  

## 2.2 样本口径

**数据来源与覆盖范围**：

| 数据类别 | 来源文件 | 年份范围 |
|----------|----------|----------|
| 公司治理、市值、回报率 | 常用变量查询（年度）.xlsx | 2000-2023 |
| 财务报表（资产、负债、利润等） | 跨表查询_沪深京股票(年频).xlsx | 2011-2024 |
| 公司基本信息（行业、上市日期） | STK_LISTEDCOINFOANL.xlsx | 最新 |

**样本筛选规则**：
- 保留 A 股上市公司年度数据
- 合并主键：`code` + `year`，采用 outer join 保留两边所有观测
- 财务数据最早仅到 2011 年，因此 2000-2010 年的杠杆率（Lev）、ROA、Cash 等指标为缺失状态

**说明**：作业要求"2000年至今"为理想目标，实际分析基于数据可得性（财务数据 2011 年起）展开。

In [10]:
# ==============================================
# 功能：2.2 样本口径
# 说明数据来源与覆盖范围，执行样本筛选
# ==============================================
import pandas as pd

print("=" * 60)
print("📋 2.2 样本口径")
print("=" * 60)

print("""
📊 数据来源与覆盖范围：

| 数据类别              | 来源文件                        | 年份范围    |
|---------------------|-------------------------------|-----------|
| 公司治理、市值、回报率  | 常用变量查询（年度）.xlsx         | 2000-2023 |
| 财务报表（资产、负债等）| 跨表查询_沪深京股票(年频).xlsx    | 2011-2024 |
| 公司基本信息          | STK_LISTEDCOINFOANL.xlsx        | 最新       |

说明：
- 作业要求"2000年至今"为理想目标
- 财务数据实际最早仅到 2011 年
- 因此 2000-2010 年的杠杆率（Lev）、ROA、Cash 等指标为缺失状态
- 后续分析时对每张图单独过滤有效值
""")

fin_y = sorted(fin_raw["year"].dropna().unique())
gov_y = sorted(gov_raw["year"].dropna().unique())
print(f"   财务数据覆盖年份：{fin_y}")
print(f"   治理数据覆盖年份：{gov_y}")

print("✅ 2.2 样本口径说明完成！")


📋 2.2 样本口径

📊 数据来源与覆盖范围：

| 数据类别              | 来源文件                        | 年份范围    |
|---------------------|-------------------------------|-----------|
| 公司治理、市值、回报率  | 常用变量查询（年度）.xlsx         | 2000-2023 |
| 财务报表（资产、负债等）| 跨表查询_沪深京股票(年频).xlsx    | 2011-2024 |
| 公司基本信息          | STK_LISTEDCOINFOANL.xlsx        | 最新       |

说明：
- 作业要求"2000年至今"为理想目标
- 财务数据实际最早仅到 2011 年
- 因此 2000-2010 年的杠杆率（Lev）、ROA、Cash 等指标为缺失状态
- 后续分析时对每张图单独过滤有效值

   财务数据覆盖年份：[np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   治理数据覆盖年份：[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int6

## 2.3 指标构造

根据作业要求，构造以下 13 个公司—年度分析变量。所有比率变量使用小数形式（如 0.35 表示 35%）。

| 变量 | 名称 | 计算公式 |
|------|------|---------|
| `Lev` | 总负债率 | 总负债 / 总资产 |
| `SL` | 流动负债率 | 流动负债 / 总资产 |
| `LL` | 长期负债率 | 非流动负债 / 总资产 |
| `SDR` | 短债比率 | 流动负债 / 总负债 |
| `Cash` | 现金比率 | 货币资金 / 总资产 |
| `ROA` | 总资产收益率 | 净利润 / 总资产 |
| `ROE` | 净资产收益率 | 净利润 / 所有者权益 |
| `SLoan` | 短期银行借款率 | 短期借款 / 总资产 |
| `LLoan` | 长期银行借款率 | 长期借款 / 总资产 |
| `Top1` | 第一大股东持股比例 | 直接取自 Shrcr1 |
| `HHI5` | 前五大股东持股集中度 | Σ(前5位股东持股比例平方) |
| `Size` | 公司规模 | ln(总资产) |
| `Age` | 上市年限 | 会计年度 - 上市年份 + 1 |

In [12]:
# ==============================================
# 功能：2.3 指标构造
# 基于 fin_raw（2.1清洗后）构造 13 个分析变量
# ==============================================
import numpy as np

print("=" * 60)
print("📐 2.3 指标构造")
print("=" * 60)

# ---------- 杠杆类 ----------
print("\n📐 构造杠杆类指标...")
fin_raw["Lev_raw"] = fin_raw["total_liability"] / fin_raw["total_asset"]
fin_raw["SL_raw"]  = fin_raw["current_liability"] / fin_raw["total_asset"]
fin_raw["LL_raw"]  = (fin_raw["total_liability"] - fin_raw["current_liability"]) / fin_raw["total_asset"]
fin_raw["SDR_raw"] = fin_raw["current_liability"] / fin_raw["total_liability"]
print("   ✅ Lev、SL、LL、SDR 构造完成")

# ---------- 盈利类 ----------
print("📐 构造盈利类指标...")
fin_raw["ROA_raw"] = fin_raw["net_profit"] / fin_raw["total_asset"]
fin_raw["ROE_raw"] = fin_raw["net_profit"] / fin_raw["equity"]
print("   ✅ ROA、ROE 构造完成")

# ---------- 现金类 ----------
print("📐 构造现金类指标...")
fin_raw["Cash_raw"] = fin_raw["cash"] / fin_raw["total_asset"]
print("   ✅ Cash 构造完成")

# ---------- 银行借款类（列可能不存在，加判断） ----------
print("📐 构造银行借款类指标（SLoan、LLoan）...")
if "short_term_borrow" in fin_raw.columns:
    fin_raw["SLoan_raw"] = fin_raw["short_term_borrow"] / fin_raw["total_asset"]
    print("   ✅ SLoan 构造完成（short_term_borrow 列存在）")
else:
    fin_raw["SLoan_raw"] = np.nan
    print("   ⚠️  short_term_borrow 列不存在，SLoan 留空")

if "long_term_borrow" in fin_raw.columns:
    fin_raw["LLoan_raw"] = fin_raw["long_term_borrow"] / fin_raw["total_asset"]
    print("   ✅ LLoan 构造完成（long_term_borrow 列存在）")
else:
    fin_raw["LLoan_raw"] = np.nan
    print("   ⚠️  long_term_borrow 列不存在，LLoan 留空")

# ---------- 合并治理数据（Top1, HHI5） ----------
print("\n🔄 合并治理数据（Top1, HHI5）...")
cols = ["code", "year", "Top1", "HHI5", "Top2_to_Top1"]
gov_sub = gov_raw[cols].drop_duplicates(subset=["code", "year"])
b = fin_raw.shape[0]
fin_raw = fin_raw.merge(gov_sub, on=["code", "year"], how="left")
print(f"   合并后：{b} → {fin_raw.shape[0]} 行")

# ---------- 构造 Size 和 Age ----------
print("📐 构造 Size 和 Age...")
fin_raw["Size"] = np.log(fin_raw["total_asset"])
info_sub = info_raw[["code", "list_year"]].drop_duplicates(subset=["code"])
fin_raw = fin_raw.merge(info_sub, on="code", how="left")
fin_raw["Age"] = fin_raw["year"] - fin_raw["list_year"] + 1
print("   ✅ Size、Age 构造完成")

# ---------- 汇总 ----------
print("\n" + "=" * 60)
print("✅ 2.3 指标构造完成！")
print("=" * 60)
vars_list = [
    "Lev_raw","SL_raw","LL_raw","SDR_raw",
    "ROA_raw","ROE_raw","Cash_raw",
    "SLoan_raw","LLoan_raw",
    "Top1","HHI5","Top2_to_Top1","Size","Age"
]
for v in vars_list:
    if v in fin_raw.columns:
        n = fin_raw[v].notna().sum()
        print(f"   {v:<20s}  非空：{n:>6d}  均值：{fin_raw[v].mean():.4f}")
    else:
        print(f"   {v:<20s}  ⚠️  不存在")
print(f"\n   数据规模：{fin_raw.shape[0]} 行 × {fin_raw.shape[1]} 列")


📐 2.3 指标构造

📐 构造杠杆类指标...
   ✅ Lev、SL、LL、SDR 构造完成
📐 构造盈利类指标...
   ✅ ROA、ROE 构造完成
📐 构造现金类指标...
   ✅ Cash 构造完成
📐 构造银行借款类指标（SLoan、LLoan）...
   ⚠️  short_term_borrow 列不存在，SLoan 留空
   ✅ LLoan 构造完成（long_term_borrow 列存在）

🔄 合并治理数据（Top1, HHI5）...
   合并后：38126 → 38126 行
📐 构造 Size 和 Age...
   ✅ Size、Age 构造完成

✅ 2.3 指标构造完成！
   Lev_raw               非空： 38126  均值：0.5344
   SL_raw                非空： 30195  均值：0.1230
   LL_raw                非空： 30195  均值：0.4589
   SDR_raw               非空： 30195  均值：0.1952
   ROA_raw               非空： 38126  均值：0.2070
   ROE_raw               非空： 38126  均值：0.0515
   Cash_raw              非空： 38046  均值：0.4923
   SLoan_raw             非空：     0  均值：nan
   LLoan_raw             非空： 31144  均值：0.0683
   Top1                  非空： 31620  均值：33.3580
   HHI5                  非空： 31620  均值：0.1539
   Top2_to_Top1          非空： 31618  均值：7.7425
   Size                  非空： 38126  均值：21.3308
   Age                   非空： 37245  均值：9.1846

   数据规模：38126 行 × 48 列


## 2.4 离群值处理（Winsorize）

对以下 10 个变量进行 1% / 99% 分位数缩尾处理：

```
Lev, SL, LL, SDR, Cash, ROA, ROE, SLoan, LLoan, HHI5
```

**规则说明**：
- 按年度分组，分别计算每个变量的 1% 和 99% 分位数
- 低于 1% 分位数的值替换为 1% 分位数；高于 99% 分位数的值替换为 99% 分位数
- `Size`、`Top1`、`Age` 一般不进行缩尾处理
- 缩尾后同时保留原始变量（`*_raw`）和缩尾变量（例如 `Lev`）

**为什么用缩尾而不是直接删除**：
- 缩尾保留了所有观测值的数量，只是调整极端值
- 删除极端值会损失样本量，且在面板数据中可能导致不平衡

In [13]:
# ==============================================
# 功能：2.4 离群值处理（Winsorize）
# 1% / 99% 分位数缩尾，按年度分组
# ==============================================
import os

print("=" * 60)
print("🪓 2.4 离群值处理（Winsorize）")
print("=" * 60)

def winsorize_by_year(df, col, low=1, high=99):
    lo = df.groupby("year")[col].transform(lambda x: x.quantile(low / 100))
    hi = df.groupby("year")[col].transform(lambda x: x.quantile(high / 100))
    return df[col].clip(lower=lo, upper=hi)

winsor_pairs = [
    ("Lev_raw","Lev"),("SL_raw","SL"),("LL_raw","LL"),("SDR_raw","SDR"),
    ("ROA_raw","ROA"),("ROE_raw","ROE"),("Cash_raw","Cash"),
    ("SLoan_raw","SLoan"),("LLoan_raw","LLoan"),
    ("HHI5","HHI5")  # HHI5 无 _raw 后缀，直接缩尾
]

print("\n缩尾变量列表：Lev, SL, LL, SDR, ROA, ROE, Cash, SLoan, LLoan, HHI5")
print("(Size、Top1、Age 不参与缩尾)\n")

stats = []
for raw_col, clean_col in winsor_pairs:
    if raw_col in fin_raw.columns:
        col = raw_col
    else:
        col = clean_col  # HHI5 等直接用原名
    if col not in fin_raw.columns:
        print(f"   ⚠️  {col} 不存在，跳过")
        continue

    bef_m, bef_s = fin_raw[col].mean(), fin_raw[col].std()
    fin_raw[clean_col] = winsorize_by_year(fin_raw, col)
    aft_m, aft_s = fin_raw[clean_col].mean(), fin_raw[clean_col].std()
    stats.append({
        "variable": clean_col,
        "before_mean": bef_m, "after_mean": aft_m,
        "before_std": bef_s, "after_std": aft_s
    })
    print(f"   ✅ {clean_col:<10s}  均值：{bef_m:.4f} → {aft_m:.4f}  标准差：{bef_s:.4f} → {aft_s:.4f}")

# 保存缩尾统计表
os.makedirs("output/tables", exist_ok=True)
winsor_df = pd.DataFrame(stats)
winsor_df["mean_change"] = winsor_df["after_mean"] - winsor_df["before_mean"]
winsor_df["std_change"]  = winsor_df["after_std"]  - winsor_df["before_std"]
winsor_df.to_csv("output/tables/winsor_summary.csv", index=False, encoding="utf-8-sig")
print(f"\n💾 缩尾统计表已保存：output/tables/winsor_summary.csv")
print("✅ 2.4 离群值处理完成！")


🪓 2.4 离群值处理（Winsorize）

缩尾变量列表：Lev, SL, LL, SDR, ROA, ROE, Cash, SLoan, LLoan, HHI5
(Size、Top1、Age 不参与缩尾)

   ✅ Lev         均值：0.5344 → 0.5345  标准差：0.2573 → 0.2568
   ✅ SL          均值：0.1230 → 0.1222  标准差：0.1314 → 0.1283
   ✅ LL          均值：0.4589 → 0.4588  标准差：0.2151 → 0.2140
   ✅ SDR         均值：0.1952 → 0.1944  标准差：0.1846 → 0.1814
   ✅ ROA         均值：0.2070 → 0.2592  标准差：1.6577 → 0.4106
   ✅ ROE         均值：0.0515 → 0.2360  标准差：7.3288 → 0.3910
   ✅ Cash        均值：0.4923 → 0.4655  标准差：0.8526 → 0.3917
   ✅ SLoan       均值：nan → nan  标准差：nan → nan
   ✅ LLoan       均值：0.0683 → 0.0676  标准差：0.1007 → 0.0974
   ✅ HHI5        均值：0.1539 → 0.1531  标准差：0.1116 → 0.1081

💾 缩尾统计表已保存：output/tables/winsor_summary.csv
✅ 2.4 离群值处理完成！


## 2.5 合并与输出

将资产负债表、利润表、股权结构、公司基本信息和行业信息合并为公司---年度面板数据。

要求：

-   合并主键为 。`code-year`
-   每次合并前后都记录行数变化。
-   在 中记录关键处理步骤，例如读取成功、合并成功、删除重复值、处理缺失值、输出文件等。`process_log.txt`
-   最终保存以下文件：

```
data/clean/firm_year_clean.csv
data/combined/csmar_firm_year_panel.csv
output/tables/missing_summary.csv
output/tables/winsor_summary.csv
```

In [19]:
# ==============================================
# 功能：2.5 合并与输出
# 合并资产负债表、利润表、股权结构、公司基本信息、行业信息
# 合并主键：code-year
# 输出文件：
#   data/clean/firm_year_clean.csv
#   data/combined/csmar_firm_year_panel.csv
#   output/tables/missing_summary.csv
#   output/tables/winsor_summary.csv
# ==============================================
import pandas as pd
import numpy as np
import os

print("=" * 60)
print("📦 2.5 合并与输出")
print("=" * 60)

# ---------- 读取原始数据 ----------
print("\n🔄 读取原始数据...")

# 1）财务数据（资产负债表 + 利润表）
fin_raw = pd.read_excel("data/raw/CSMAR/data_raw_zip/跨表查询_沪深京股票(年频).xlsx")
fin_rename = {}
for col in fin_raw.columns:
    if col == "code": fin_rename[col] = "code"
    elif "enddate" in col.lower(): fin_rename[col] = "end_date"
    elif "A001100000" in col: fin_rename[col] = "total_asset"
    elif "A001101000" in col: fin_rename[col] = "current_asset"
    elif "A002000000" in col: fin_rename[col] = "total_liability"
    elif "A002101000" in col: fin_rename[col] = "current_liability"
    elif "A002200000" in col: fin_rename[col] = "noncurrent_liability"
    elif "A002206000" in col: fin_rename[col] = "long_term_borrow"
    elif "A003101000" in col: fin_rename[col] = "revenue"
    elif "A003105000" in col: fin_rename[col] = "net_profit"
    elif "A003102000" in col: fin_rename[col] = "cash"
    elif "A003000000" in col: fin_rename[col] = "equity"
fin_raw = fin_raw.rename(columns=fin_rename)
fin_raw["code"] = fin_raw["code"].astype(str).str.strip().str.zfill(6)
fin_raw["year"] = pd.to_datetime(fin_raw["end_date"], errors="coerce").dt.year
fin_raw["year"] = fin_raw["year"].astype("Int64")
print(f"   财务（资产负债表+利润表）：{fin_raw.shape[0]} 行 × {fin_raw.shape[1]} 列")

# 2）治理数据（股权结构）
gov_raw = pd.read_excel("data/raw/CSMAR/data_raw_zip/常用变量查询（年度）.xlsx", header=0)
gov_raw = gov_raw.iloc[1:].reset_index(drop=True)
gov_raw = gov_raw.rename(columns={
    "Stkcd": "code", "accper": "year",
    "Shrcr1": "Top1", "Shrhfd5": "HHI5", "Shrz": "Top2_to_Top1"
})
gov_raw = gov_raw[gov_raw["code"] != "股票代码"]
gov_raw["code"] = gov_raw["code"].astype(str).str.strip().str.zfill(6)
gov_raw["year"] = pd.to_numeric(gov_raw["year"], errors="coerce").astype("Int64")
print(f"   治理（股权结构）：{gov_raw.shape[0]} 行 × {gov_raw.shape[1]} 列")

# 3）公司基本信息（行业信息）
info_raw = pd.read_excel("data/raw/CSMAR/data_raw_zip/STK_LISTEDCOINFOANL.xlsx")
info_rename = {}
for col in info_raw.columns:
    if "symbol" in col.lower(): info_rename[col] = "code"
    elif "shortname" in col.lower(): info_rename[col] = "stknme"
    elif "industrycode" in col.lower(): info_rename[col] = "industry_code"
    elif "industryname" in col.lower(): info_rename[col] = "industry_name"
    elif "province" in col.lower(): info_rename[col] = "province"
    elif "listingdate" in col.lower(): info_rename[col] = "listing_date"
info_raw = info_raw.rename(columns=info_rename)
info_raw["code"] = info_raw["code"].astype(str).str.strip().str.zfill(6)
info_raw["listing_date"] = pd.to_datetime(info_raw["listing_date"], errors="coerce")
info_raw["list_year"] = info_raw["listing_date"].dt.year
info_raw = info_raw.drop_duplicates(subset=["code"])
print(f"   公司信息（行业信息）：{info_raw.shape[0]} 行 × {info_raw.shape[1]} 列")

# ---------- 删除缺失行和重复行 ----------
print("\n🧹 删除缺失行和重复行...")

def clean_df(df, name):
    b = df.shape[0]
    df = df.dropna(subset=["code", "year"])
    df = df.drop_duplicates(subset=["code", "year"])
    print(f"   {name}：{b} → {df.shape[0]} 行")
    return df

fin_raw = clean_df(fin_raw, "财务数据")
gov_raw = clean_df(gov_raw, "治理数据")

# ---------- 合并面板（code-year 为主键） ----------
print("\n🔄 合并面板数据（code-year 为主键）...")

# Step1：财务 × 治理（outer join）
panel = fin_raw.merge(
    gov_raw[["code", "year", "Top1", "HHI5", "Top2_to_Top1"]],
    on=["code", "year"], how="outer"
)
print(f"   财务×治理（outer）：{panel.shape[0]} 行")

# Step2：× 公司基本信息
panel = panel.merge(
    info_raw[["code", "stknme", "industry_code", "industry_name", "province", "list_year"]],
    on="code", how="left"
)
print(f"   ×公司信息：{panel.shape[0]} 行")

# Step3：过滤 total_asset 为 0 的行（防止除零）
n_zero = (panel["total_asset"] == 0).sum()
if n_zero > 0:
    panel = panel[panel["total_asset"] != 0]
    print(f"   ⚠️  删除 {n_zero} 条总资产为0的行")
else:
    print("   ✅ 无总资产为0的异常行")

# Step4：强制转数值型 + 构造分析变量
print("📐 构造分析变量...")

num_cols = [
    "total_asset", "current_asset", "total_liability", "current_liability",
    "noncurrent_liability", "long_term_borrow", "revenue",
    "net_profit", "cash", "equity"
]
for col in num_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel = panel[(panel["total_asset"].notna()) & (panel["total_asset"] != 0)]

panel["Lev_raw"]   = panel["total_liability"] / panel["total_asset"]
panel["SL_raw"]    = panel["current_liability"] / panel["total_asset"]
panel["LL_raw"]    = (panel["total_liability"] - panel["current_liability"]) / panel["total_asset"]
panel["SDR_raw"]   = panel["current_liability"] / panel["total_liability"]
panel["ROA_raw"]   = panel["net_profit"] / panel["total_asset"]
panel["ROE_raw"]   = panel["net_profit"] / panel["equity"]
panel["Cash_raw"]  = panel["cash"] / panel["total_asset"]
panel["Size"]      = np.log(panel["total_asset"])
panel["Age"]       = panel["year"] - panel["list_year"] + 1
print("   ✅ Lev、SL、LL、SDR、ROA、ROE、Cash、Size、Age 构造完成")

# Step5：缩尾处理（Winsorize 1%/99%）
print("🪓 缩尾处理（1%/99% 分位数，按年度）...")

def winsorize_by_year(df, col, low=1, high=99):
    lo = df.groupby("year")[col].transform(lambda x: x.quantile(low / 100))
    hi = df.groupby("year")[col].transform(lambda x: x.quantile(high / 100))
    return df[col].clip(lower=lo, upper=hi)

winsor_pairs = [
    ("Lev_raw", "Lev"), ("SL_raw", "SL"), ("LL_raw", "LL"), ("SDR_raw", "SDR"),
    ("ROA_raw", "ROA"), ("ROE_raw", "ROE"), ("Cash_raw", "Cash"),
    ("HHI5", "HHI5")
]
winsor_stats = []
for raw_col, clean_col in winsor_pairs:
    src = raw_col if raw_col in panel.columns else clean_col
    if src not in panel.columns:
        print(f"   ⚠️  {src} 不存在，跳过")
        continue
    bef = panel[src].mean()
    panel[clean_col] = winsorize_by_year(panel, src)
    aft = panel[src].mean()
    winsor_stats.append({"variable": clean_col, "before_mean": bef, "after_mean": aft})
    print(f"   ✅ {clean_col} 缩尾完成")

# ---------- 输出文件 ----------
print("\n💾 输出文件...")

os.makedirs("data/combined", exist_ok=True)
os.makedirs("data/clean", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

# 1）缺失值统计
miss = panel.isnull().sum().to_frame("missing_count")
miss["missing_pct"] = (miss["missing_count"] / len(panel) * 100).round(2)
miss.to_csv("output/tables/missing_summary.csv", encoding="utf-8-sig")
print(f"   ✅ output/tables/missing_summary.csv")

# 2）缩尾统计
winsor_df = pd.DataFrame(winsor_stats)
winsor_df["mean_change"] = winsor_df["after_mean"] - winsor_df["before_mean"]
winsor_df.to_csv("output/tables/winsor_summary.csv", index=False, encoding="utf-8-sig")
print(f"   ✅ output/tables/winsor_summary.csv")

# 3）治理单表（只选 panel 中实际存在的列）
select_cols = [
    "code", "year", "stknme", "Top1", "HHI5", "Top2_to_Top1",
    "Size", "Age", "industry_code", "industry_name", "province"
]
select_cols = [c for c in select_cols if c in panel.columns]
print(f"   治理单表实际列：{select_cols}")
firm_clean = panel[select_cols].drop_duplicates()
firm_clean.to_csv("data/clean/firm_year_clean.csv", index=False, encoding="utf-8-sig")
print(f"   ✅ data/clean/firm_year_clean.csv")

# 4）完整面板
panel.to_csv("data/combined/csmar_firm_year_panel.csv", index=False, encoding="utf-8-sig")
print(f"   ✅ data/combined/csmar_firm_year_panel.csv")

# ---------- 记录 process_log ----------
log = f"""
{'='*60}
时间：{pd.Timestamp.now()}
{'='*60}
读取数据：
  财务（资产负债表+利润表）：{fin_raw.shape[0]} 行
  治理（股权结构）：{gov_raw.shape[0]} 行
  公司信息（行业）：{info_raw.shape[0]} 行
删除缺失/重复后：
  财务：{fin_raw.shape[0]} 行
  治理：{gov_raw.shape[0]} 行
合并面板：{panel.shape[0]} 行 × {panel.shape[1]} 列
输出文件：
  data/combined/csmar_firm_year_panel.csv
  data/clean/firm_year_clean.csv
  output/tables/missing_summary.csv
  output/tables/winsor_summary.csv
"""
with open("process_log.txt", "a", encoding="utf-8") as f:
    f.write(log)
print("   ✅ process_log.txt 已更新")

print("\n" + "=" * 60)
print("✅ 2.5 合并与输出完成！")
print("=" * 60)
print(f"   最终面板：{panel.shape[0]} 行 × {panel.shape[1]} 列")


📦 2.5 合并与输出

🔄 读取原始数据...
   财务（资产负债表+利润表）：81665 行 × 33 列
   治理（股权结构）：61457 行 × 33 列
   公司信息（行业信息）：5615 行 × 41 列

🧹 删除缺失行和重复行...
   财务数据：81665 → 81662 行
   治理数据：61457 → 61456 行

🔄 合并面板数据（code-year 为主键）...
   财务×治理（outer）：97268 行
   ×公司信息：97268 行
   ⚠️  删除 168 条总资产为0的行
📐 构造分析变量...
   ✅ Lev、SL、LL、SDR、ROA、ROE、Cash、Size、Age 构造完成
🪓 缩尾处理（1%/99% 分位数，按年度）...
   ✅ Lev 缩尾完成
   ✅ SL 缩尾完成
   ✅ LL 缩尾完成
   ✅ SDR 缩尾完成
   ✅ ROA 缩尾完成
   ✅ ROE 缩尾完成
   ✅ Cash 缩尾完成
   ✅ HHI5 缩尾完成

💾 输出文件...
   ✅ output/tables/missing_summary.csv
   ✅ output/tables/winsor_summary.csv
   治理单表实际列：['code', 'year', 'Top1', 'HHI5', 'Top2_to_Top1', 'Size', 'Age', 'industry_code', 'industry_name', 'province']
   ✅ data/clean/firm_year_clean.csv
   ✅ data/combined/csmar_firm_year_panel.csv
   ✅ process_log.txt 已更新

✅ 2.5 合并与输出完成！
   最终面板：53746 行 × 62 列
